# Comp Confidence Scoring

A small notebook that scores how much you should trust each comp before letting it influence a price.

Two sales that happen to agree is not a price. Forty sales clustered in a tight range is. Most repricers treat both as equally informative. This notebook does not.

## What it does

For each SKU and condition grade, the notebook computes a confidence score from three inputs.

1. Volume. How many comps you actually have over the lookback window.
2. Recency. How stale the comps are. Weighted so a sale from yesterday counts more than one from 8 weeks ago.
3. Dispersion. How tight the cluster is once outliers are trimmed. A wide spread means the market is not telling you a clear story.

Each input is scored 0 to 1. The three are combined into an overall confidence score, also 0 to 1, with a plain language tier (High, Medium, Low, Do Not Use).

The output also flags individual comps as likely outliers, so you can see what the system is choosing to ignore.

## How to use it

Run all cells. The notebook generates a simulated dataset of refurbished phone and tablet sales across Amazon Renewed, Back Market, eBay, and Swappa, scores each SKU and grade, and writes an Excel file with the results.

To run on your own data, replace the `df` in the data loading cell with a DataFrame containing these columns.

Required:
- `model` (e.g. "iPhone 13 Pro")
- `grade` (e.g. "Good", "Very Good", "Excellent")
- `channel` (e.g. "Amazon Renewed")
- `sale_price` (numeric)
- `sale_date` (date or datetime)

Optional but useful:
- `seller_review_count` (numeric). Used to soft-flag listings from very low review count sellers.

## What is not in here

The actual pricing decision. This scores inputs. What you do with a Low confidence cluster, whether you hold price, defer to channel floor, or escalate, is a policy question, and those policies are where the real engagement work lives.


In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

rng = np.random.default_rng(7)

TODAY = pd.Timestamp("2026-05-12")
LOOKBACK_DAYS = 90

## Simulated data

Builds a dataset of around 600 sales across a handful of SKUs, grades, and channels. A few SKU and grade combinations have plenty of clean comps. A few have only one or two. A few have planted outliers, stale clusters, and low review count sellers, so you can see what the scoring picks up.

Skip this cell if running on your own data. Just load it into `df` instead.

In [2]:
MODELS = {
    "iPhone 13 Pro": {"Good": 505, "Very Good": 560, "Excellent": 610},
    "iPhone 14":     {"Good": 470, "Very Good": 525, "Excellent": 580},
    "Galaxy S22":    {"Good": 310, "Very Good": 360, "Excellent": 410},
    "Galaxy S23":    {"Good": 410, "Very Good": 470, "Excellent": 530},
    "iPad Air":      {"Good": 340, "Very Good": 395, "Excellent": 450},
    "Pixel 7":       {"Good": 240, "Very Good": 285, "Excellent": 330},
}

CHANNELS = ["Amazon Renewed", "Back Market", "eBay", "Swappa"]

rows = []

for model, grades in MODELS.items():
    for grade, center in grades.items():

        # Most SKU/grade combos get a healthy number of comps.
        # A handful get starved on purpose so the volume score reflects it.
        if model == "Pixel 7" and grade == "Excellent":
            n_sales = 2
        elif model == "Galaxy S22" and grade == "Good":
            n_sales = 4
        else:
            n_sales = rng.integers(25, 55)

        for _ in range(n_sales):
            channel = rng.choice(CHANNELS)
            channel_adj = {"Amazon Renewed": 1.05, "Back Market": 1.02, "eBay": 0.97, "Swappa": 1.00}[channel]
            noise = rng.normal(0, center * 0.04)
            price = round(center * channel_adj + noise, 2)

            days_ago = int(rng.integers(0, LOOKBACK_DAYS))
            sale_date = TODAY - timedelta(days=days_ago)

            seller_reviews = int(rng.integers(50, 5000))

            rows.append({
                "model": model,
                "grade": grade,
                "channel": channel,
                "sale_price": price,
                "sale_date": sale_date,
                "seller_review_count": seller_reviews,
            })

df = pd.DataFrame(rows)

# Plant some artifacts so the tool has something to find.

# 1. A wildly high outlier on a low review count seller.
df.loc[len(df)] = {
    "model": "iPhone 13 Pro", "grade": "Good", "channel": "eBay",
    "sale_price": 749.00, "sale_date": TODAY - timedelta(days=3),
    "seller_review_count": 12,
}

# 2. A cluster of stale Galaxy S23 Very Good comps. Old enough to be misleading.
for _ in range(8):
    df.loc[len(df)] = {
        "model": "Galaxy S23", "grade": "Very Good", "channel": "Back Market",
        "sale_price": round(540 + rng.normal(0, 8), 2),
        "sale_date": TODAY - timedelta(days=int(rng.integers(75, 89))),
        "seller_review_count": int(rng.integers(200, 2000)),
    }

# 3. An iPad Air Excellent cluster with wide dispersion. Real category but messy market.
for _ in range(15):
    df.loc[len(df)] = {
        "model": "iPad Air", "grade": "Excellent",
        "channel": rng.choice(CHANNELS),
        "sale_price": round(450 + rng.normal(0, 70), 2),
        "sale_date": TODAY - timedelta(days=int(rng.integers(0, 60))),
        "seller_review_count": int(rng.integers(50, 3000)),
    }

df["sale_date"] = pd.to_datetime(df["sale_date"])
df = df.reset_index(drop=True)

print(f"Loaded {len(df)} comps across {df['model'].nunique()} models.")
df.head()

Loaded 690 comps across 6 models.


,model,grade,channel,sale_price,sale_date,seller_review_count
0,iPhone 13 Pro,Good,eBay,495.88,2026-03-21,3889
1,iPhone 13 Pro,Good,Swappa,495.82,2026-04-22,1461
2,iPhone 13 Pro,Good,Swappa,506.21,2026-03-29,4115
3,iPhone 13 Pro,Good,Amazon Renewed,517.72,2026-03-02,4091
4,iPhone 13 Pro,Good,Back Market,522.31,2026-03-09,1311


## Outlier flagging

Per SKU and grade, flag comps that sit far outside the cluster. Using the IQR method, which is robust to a few extreme values and does not assume the prices are nicely bell shaped. Quick refresher in case it's been a while. The IQR is the spread of the middle half of the data. Anything more than 1.5 times that spread above or below the middle half gets flagged.

Also soft-flag anything from a very low review count seller. Not because the price is necessarily wrong, but because it shouldn't carry the same weight as a comp from a high volume seller. One listing from a 12-review account is, in the language of the post, an artifact.

In [3]:
def iqr_bounds(s):
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    return pd.Series({"lo": q1 - 1.5 * iqr, "hi": q3 + 1.5 * iqr})

bounds = df.groupby(["model", "grade"])["sale_price"].agg([
    ("lo", lambda s: s.quantile(0.25) - 1.5 * (s.quantile(0.75) - s.quantile(0.25))),
    ("hi", lambda s: s.quantile(0.75) + 1.5 * (s.quantile(0.75) - s.quantile(0.25))),
]).reset_index()

df = df.merge(bounds, on=["model", "grade"], how="left")
df["is_price_outlier"] = (df["sale_price"] < df["lo"]) | (df["sale_price"] > df["hi"])
df = df.drop(columns=["lo", "hi"])

df["is_low_reviews"] = df["seller_review_count"] < 50
df["is_flagged"] = df["is_price_outlier"] | df["is_low_reviews"]

print(f"Flagged {df['is_flagged'].sum()} of {len(df)} comps as outliers or low-review.")
df[df["is_flagged"]].head(10)

Flagged 8 of 690 comps as outliers or low-review.


,model,grade,channel,sale_price,sale_date,seller_review_count,is_price_outlier,is_low_reviews,is_flagged
10,iPhone 13 Pro,Good,eBay,439.01,2026-03-01,4589,True,False,True
207,iPhone 14,Excellent,Amazon Renewed,652.31,2026-04-11,1481,True,False,True
279,Galaxy S22,Very Good,Amazon Renewed,414.47,2026-04-28,4418,True,False,True
474,iPad Air,Good,eBay,300.04,2026-05-05,216,True,False,True
663,Pixel 7,Very Good,Back Market,315.51,2026-03-08,877,True,False,True
666,iPhone 13 Pro,Good,eBay,749.00,2026-05-09,12,True,True,True
683,iPad Air,Excellent,Swappa,300.54,2026-04-27,218,True,False,True
689,iPad Air,Excellent,eBay,307.38,2026-04-18,1925,True,False,True


## Confidence scoring

Three sub-scores, each 0 to 1.

Volume score. Hits 1.0 at 30 clean comps, 0 at zero, scales linearly in between. Thirty is a working threshold, not gospel. Adjust to your category.

Recency score. The average age of clean comps in days. 0 days old is 1.0, 60+ days old is 0.0. So a cluster that's mostly from the last two weeks scores high. A cluster from 8 weeks ago scores near zero, even if it has plenty of points.

Dispersion score. The coefficient of variation, the standard deviation as a percentage of the mean, on the clean comps. Coefficient of variation is shorthand for how wide the spread is relative to the typical price. 5 percent or tighter scores 1.0. 25 percent or wider scores 0.0.

Overall confidence is a weighted average. Volume and dispersion weighted slightly higher than recency, because a stale but enormous and tight cluster is still useful information, with a caveat.

In [4]:
def score_cluster(group):
    clean = group[~group["is_flagged"]]
    n = len(clean)

    if n == 0:
        return pd.Series({
            "n_total": len(group),
            "n_clean": 0,
            "median_price": np.nan,
            "price_low": np.nan,
            "price_high": np.nan,
            "avg_age_days": np.nan,
            "cv": np.nan,
            "vol_score": 0.0,
            "recency_score": 0.0,
            "dispersion_score": 0.0,
            "confidence": 0.0,
        })

    median_price = clean["sale_price"].median()
    p10, p90 = clean["sale_price"].quantile([0.1, 0.9])
    mean_price = clean["sale_price"].mean()
    std_price = clean["sale_price"].std() if n > 1 else 0.0
    cv = (std_price / mean_price) if mean_price else np.nan

    ages = (TODAY - clean["sale_date"]).dt.days
    avg_age = ages.mean()

    vol_score = min(n / 30.0, 1.0)
    recency_score = max(0.0, 1.0 - (avg_age / 60.0))

    # CV of 0.05 or lower is great, 0.25 or higher is uninformative.
    if pd.isna(cv):
        dispersion_score = 0.0
    else:
        dispersion_score = max(0.0, min(1.0, (0.25 - cv) / 0.20))

    confidence = (
        0.4 * vol_score +
        0.25 * recency_score +
        0.35 * dispersion_score
    )

    return pd.Series({
        "n_total": len(group),
        "n_clean": n,
        "median_price": round(median_price, 2),
        "price_low": round(p10, 2),
        "price_high": round(p90, 2),
        "avg_age_days": round(avg_age, 1),
        "cv": round(cv, 3) if not pd.isna(cv) else np.nan,
        "vol_score": round(vol_score, 2),
        "recency_score": round(recency_score, 2),
        "dispersion_score": round(dispersion_score, 2),
        "confidence": round(confidence, 2),
    })

summary = (
    df.groupby(["model", "grade"])
      .apply(score_cluster, include_groups=False)
      .reset_index()
)

def tier(c):
    if c >= 0.75:
        return "High"
    if c >= 0.5:
        return "Medium"
    if c >= 0.25:
        return "Low"
    return "Do Not Use"

summary["tier"] = summary["confidence"].apply(tier)
summary = summary.sort_values("confidence", ascending=False).reset_index(drop=True)
summary

,model,grade,n_total,n_clean,median_price,price_low,price_high,avg_age_days,cv,vol_score,recency_score,dispersion_score,confidence,tier
0,iPhone 14,Very Good,50.0,50.0,522.52,493.33,553.80,41.4,0.044,1.00,0.31,1.00,0.83,High
1,Pixel 7,Very Good,37.0,36.0,284.71,268.21,297.75,44.1,0.037,1.00,0.27,1.00,0.82,High
2,iPhone 13 Pro,Excellent,32.0,32.0,609.83,579.30,656.21,44.9,0.052,1.00,0.25,0.99,0.81,High
3,Galaxy S22,Excellent,52.0,52.0,417.12,393.24,443.14,45.2,0.051,1.00,0.25,1.00,0.81,High
4,Galaxy S23,Good,46.0,46.0,408.02,385.71,429.34,45.7,0.046,1.00,0.24,1.00,0.81,High
5,Galaxy S23,Excellent,29.0,29.0,526.52,511.93,568.99,41.7,0.044,0.97,0.31,1.00,0.81,High
6,iPhone 13 Pro,Good,54.0,52.0,505.52,475.16,530.38,46.8,0.042,1.00,0.22,1.00,0.81,High
7,iPhone 14,Excellent,46.0,45.0,581.64,560.43,611.06,46.5,0.035,1.00,0.22,1.00,0.81,High
8,iPad Air,Very Good,39.0,39.0,401.36,375.65,429.90,48.1,0.052,1.00,0.20,0.99,0.80,High
9,Galaxy S22,Very Good,52.0,51.0,358.40,335.14,381.15,46.8,0.053,1.00,0.22,0.98,0.80,High


## What the scoring caught

Sanity check, in plain language.

In [5]:
print("High confidence clusters (you can build rules on these):")
print(summary[summary["tier"] == "High"][["model", "grade", "n_clean", "median_price", "avg_age_days", "confidence"]].to_string(index=False))

print("\nLow confidence or unusable clusters (where blind trust gets you in trouble):")
print(summary[summary["tier"].isin(["Low", "Do Not Use"])][["model", "grade", "n_clean", "median_price", "avg_age_days", "cv", "confidence", "tier"]].to_string(index=False))

print("\nFlagged individual comps the system is choosing to ignore:")
flagged = df[df["is_flagged"]][["model", "grade", "channel", "sale_price", "sale_date", "seller_review_count", "is_price_outlier", "is_low_reviews"]]
print(flagged.head(15).to_string(index=False))

High confidence clusters (you can build rules on these):
        model     grade  n_clean  median_price  avg_age_days  confidence
    iPhone 14 Very Good     50.0        522.52          41.4        0.83
      Pixel 7 Very Good     36.0        284.71          44.1        0.82
iPhone 13 Pro Excellent     32.0        609.83          44.9        0.81
   Galaxy S22 Excellent     52.0        417.12          45.2        0.81
   Galaxy S23      Good     46.0        408.02          45.7        0.81
   Galaxy S23 Excellent     29.0        526.52          41.7        0.81
iPhone 13 Pro      Good     52.0        505.52          46.8        0.81
    iPhone 14 Excellent     45.0        581.64          46.5        0.81
     iPad Air Very Good     39.0        401.36          48.1        0.80
   Galaxy S22 Very Good     51.0        358.40          46.8        0.80
    iPhone 14      Good     40.0        480.30          49.3        0.79
     iPad Air      Good     51.0        340.21          50.9       

## Excel output

Two sheets. One with the per SKU and grade scoring. One with the individual comps and their flags. Refurbishers live in Excel, so this is the actual deliverable.

In [6]:
out_path = "comp_confidence_output.xlsx"

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="Cluster Scores", index=False)
    df.to_excel(writer, sheet_name="Comps With Flags", index=False)

print(f"Wrote {out_path}")

Wrote comp_confidence_output.xlsx


## Notes on what this is and isn't

This scores the inputs. It does not set prices, pick channels, or decide what to do when confidence is Low. Those are policy questions, and the right policy depends on your margin structure, channel mix, return rates, and how long your aged inventory takes to bleed.

The thresholds (30 clean comps for full volume credit, 60 day recency window, 5 percent and 25 percent CV bounds) are working defaults. Real engagements tune these to the category and the channel. iPad accessories don't behave like flagship phones, and Amazon Renewed doesn't behave like eBay.

The methods that handle seasonality, release-cycle decay, and channel drift sit on top of this. None of them work well if the inputs aren't filtered first, which is the entire point of the post.
